# 04 — Exploitation des résultats de requêtes

**C'est le notebook le plus important du repo.**

Une fois qu'on a récupéré des données (notebook 03), la vraie valeur commence :
**qu'est-ce qu'on FAIT avec le résultat ?**

Ce notebook couvre le pipeline complet d'exploitation :

1. Conversion du résultat brut → `pandas.DataFrame`
2. Nettoyage / transformation
3. Analyse (statistiques, agrégations, anomalies)
4. Visualisation (graphiques statiques et interactifs)
5. Export (CSV, Excel, JSON, réinjection en base)


In [ ]:
import sys
sys.path.append("..")

from db_connection.sqlite_connector import get_sqlite_engine
from sqlalchemy import text
from queries.write_queries import bulk_insert

engine = get_sqlite_engine()

with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS demo_sales (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            product TEXT,
            category TEXT,
            quantity INTEGER,
            unit_price REAL,
            sale_date TEXT
        )
    '''))

bulk_insert(engine, "demo_sales", [
    {"product": "Clavier", "category": "Informatique", "quantity": 3, "unit_price": 89.99, "sale_date": "2026-01-05"},
    {"product": "Souris", "category": "Informatique", "quantity": 10, "unit_price": 29.99, "sale_date": "2026-01-06"},
    {"product": "Chaise", "category": "Mobilier", "quantity": 1, "unit_price": 149.0, "sale_date": "2026-01-07"},
    {"product": "Écran", "category": "Informatique", "quantity": 2, "unit_price": 249.0, "sale_date": "2026-01-08"},
    {"product": "Bureau", "category": "Mobilier", "quantity": 1, "unit_price": 320.0, "sale_date": "2026-01-09"},
])
print("Données de démo prêtes.")


## Étape 1 — Récupérer et convertir le résultat en DataFrame

`query_to_dataframe()` fait tout en une fois : exécute la requête ET convertit
le résultat en DataFrame pandas, prêt à être manipulé.


In [ ]:
from data_processing.result_to_dataframe import query_to_dataframe

df = query_to_dataframe(engine, "SELECT * FROM demo_sales")
df


## Étape 2 — Nettoyer et transformer

Avant toute analyse, on nettoie et on type correctement les données
(le SQL renvoie parfois du texte là où on veut un nombre ou une date).


In [ ]:
from data_processing.clean_transform import drop_duplicates_and_nulls, cast_types, add_computed_column

df = drop_duplicates_and_nulls(df)
df = cast_types(df, {"sale_date": "datetime64[ns]", "quantity": "int64", "unit_price": "float64"})

# Colonne calculée : chiffre d'affaires par ligne
df = add_computed_column(df, "total", lambda row: row["quantity"] * row["unit_price"])
df


## Étape 3 — Analyser le résultat

Statistiques descriptives, agrégations par groupe, top N, détection d'anomalies.


In [ ]:
from data_processing.analyze_results import summary_statistics, group_and_aggregate, top_n

stats = summary_statistics(df, "total")
stats


In [ ]:
ca_par_categorie = group_and_aggregate(df, group_by="category", agg_column="total", agg_func="sum")
ca_par_categorie


In [ ]:
top_ventes = top_n(df, column="total", n=3)
top_ventes


## Étape 4 — Visualiser le résultat

Deux options : `matplotlib` (statique, pour rapport) ou `plotly` (interactif, pour notebook/dashboard).


In [ ]:
from data_processing.visualize_results import plot_bar_matplotlib

plot_bar_matplotlib(ca_par_categorie, x="category", y="total", title="Chiffre d'affaires par catégorie")


In [ ]:
from data_processing.visualize_results import plot_interactive

fig = plot_interactive(df, x="product", y="total", kind="bar", title="CA par produit (interactif)")


## Étape 5 — Exporter le résultat exploité

Dernière étape : sortir le résultat sous une forme finale utilisable
(fichier partageable, ou réinjection dans une autre table).


In [ ]:
from data_processing.export_results import export_to_csv, export_to_excel, export_to_json

export_to_csv(ca_par_categorie, "../exports/ca_par_categorie.csv")
export_to_excel(ca_par_categorie, "../exports/ca_par_categorie.xlsx")
export_to_json(ca_par_categorie, "../exports/ca_par_categorie.json")

print("Exports terminés — voir le dossier /exports")


In [ ]:
from data_processing.export_results import export_to_table

# Réinjection du résultat traité dans une nouvelle table (pipeline complet)
export_to_table(ca_par_categorie, engine, table_name="rapport_ca_categorie", if_exists="replace")
print("Résultat réinjecté dans la table 'rapport_ca_categorie'.")


## Ce qu'il faut retenir

> Le résultat brut d'une requête (liste de tuples/dicts) n'est presque jamais
> le point final. Le vrai travail commence après : **nettoyer → analyser →
> visualiser → exporter/agir**. C'est ce pipeline qui transforme une requête
> SQL en une information ou une action exploitable.

## Suite

➡️ Notebook **`05_supabase_specifique.ipynb`** pour les fonctionnalités propres à Supabase
(RLS, Realtime, Storage).

➡️ Notebook **`06_projet_complet_pipeline.ipynb`** pour un exemple de bout en bout
combinant TOUT (connexion → requêtage → exploitation → action).
